# IPL Team Performance: analysis notebook

This notebook walks through data quality, then the main business questions, then conclusions. It reads the files in `output/`, so run `data_processing.py` first.

## Part 1: data quality

In [ ]:
import sys, json
from pathlib import Path
OUT = Path.cwd().parent / "output"
sys.path.insert(0, str(Path.cwd().parent))
print((OUT / "data_quality_report.md").read_text(encoding="utf-8"))

In [ ]:
summary = json.loads((OUT / "cleaning_summary.json").read_text())
for table, info in summary["tables"].items():
    print(f"{table}: {info['raw_rows']:,} raw rows, {info['clean_rows']:,} clean rows, "
          f"{info['duplicates_removed']} duplicates removed")
print("Outliers flagged (kept, not deleted):", summary["outliers"]["flagged_rows"])

## Part 2: business questions

Each cell answers one question. The figures come from the same functions the dashboard uses.

### Which team wins the largest share of its decided matches?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from data_processing import team_kpis, phase_table, season_order, MIN_DECIDED_RELIABLE

matches = pd.read_csv(OUT / "cleaned_matches.csv", dtype={"season": str}, parse_dates=["date"])
balls = pd.read_csv(OUT / "cleaned_ball_by_ball.csv", dtype={"season": str}, low_memory=False)
kpis = team_kpis(matches, balls)

table = kpis[["matches_played", "decided_matches", "wins", "win_rate_pct"]].sort_values("win_rate_pct", ascending=False)
table

In [ ]:
k = kpis.sort_values("win_rate_pct")
colors = ["#2a6f97" if d >= MIN_DECIDED_RELIABLE else "#bbbbbb" for d in k["decided_matches"]]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(k.index, k["win_rate_pct"], color=colors)
ax.set_xlabel("Win rate (% of decided matches)")
ax.set_title("Grey = fewer than 20 decided matches")
plt.tight_layout(); plt.show()

### Does winning the toss help?

In [ ]:
toss = kpis[["decided_after_toss_win", "win_rate_after_toss_win_pct",
             "decided_after_toss_loss", "win_rate_after_toss_loss_pct"]]
toss_all = matches[matches["winner"].notna()]
overall = (toss_all["toss_winner"] == toss_all["winner"]).mean() * 100
print(f"Across all decided matches, the toss winner won {overall:.1f}% of the time.")
toss

### How fast do teams score in each phase, season by season?

In [ ]:
pt = phase_table(balls)
death = pt[pt["phase"] == "Death"].groupby(["season"])[["runs", "legal_balls"]].sum()
death["death_runs_per_100_balls"] = (100 * death["runs"] / death["legal_balls"]).round(1)
season_list = season_order(matches)
death.reindex(season_list)[["death_runs_per_100_balls"]]

In [ ]:
rates = (pt.groupby(["season", "phase"])[["runs", "legal_balls"]].sum()
         .assign(rate=lambda d: 100 * d["runs"] / d["legal_balls"]).reset_index())
fig, ax = plt.subplots(figsize=(10, 4.5))
for phase in ["Powerplay", "Middle", "Death"]:
    s = rates[rates["phase"] == phase].set_index("season")["rate"].reindex(season_list)
    ax.plot(season_list, s.values, marker="o", label=phase)
ax.set_ylabel("Runs per 100 balls")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout(); plt.show()

### How are batters dismissed?

In [ ]:
d = balls[balls["counts_as_dismissal"] == 1]["wicket_kind"].value_counts(normalize=True).mul(100).round(1)
d.rename("share_pct").to_frame()

## Part 3: conclusions

**Win rates.** Gujarat Titans have the highest rate, 60.9% of decided matches, but on only 69 matches because they entered in 2022. Chennai Super Kings (56.2%) and Mumbai Indians (54.5%) lead over a long record. Pune Warriors have the lowest rate, 26.7%, on 45 decided matches.

**Toss.** Toss winners won 51.6% of decided matches, which is close to even. The choice made after winning the toss matters more: teams that chose to field won 54.6% of those matches, and teams that chose to bat won 45.8%.

**Scoring.** Death overs run fastest, at 167 runs per 100 balls, against 134 in the powerplay and 132 in the middle overs. The death-overs rate rose from 157 in 2021 to 187 in 2024.

**Dismissals.** Catches account for 63.3% of dismissals. Bowled, run out and lbw make up most of the rest.

**Caveats.** Win rates do not adjust for opposition strength. The 25 no-result matches are excluded from win rates. The 2026 season is still in progress in this data.